# IF Task-Vector Grad-Norm Top-k Sparse Updates (1%, 10%, 50%, 100%)

This notebook first runs the original IF task-vector magnitude sparse update experiment:

- `theta_sparse = theta_base + mask * (theta_if - theta_base)`
- `mask = 1[|Delta_if| >= threshold]`
- `Delta_if = theta_if - theta_base`

Then an **appended Fisher-based cell** runs the same sparse update formula with
`mask = 1[F_if >= threshold]` using the precomputed IF Fisher artifact.

In [13]:
from __future__ import annotations

import gc
import json
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Mapping, Tuple

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration for IF sparse-update experiments.

    Args:
        base_model_id: Base model id/path used as sparse-update anchor.
        if_model_path: IF-tuned model checkpoint path that provides IF task vector.
        output_root: Root output directory for sparse checkpoints and metadata.
        model_dtype_name: Model loading dtype alias (`bf16`, `fp16`, or `fp32`).
        seed: RNG seed for reproducible threshold sampling.
    """

    base_model_id: str
    if_model_path: Path
    output_root: Path
    model_dtype_name: str
    seed: int


RUNTIME = RuntimeConfig(
    base_model_id='Qwen/Qwen3-1.7B',
    if_model_path=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface'),
    output_root=Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge'),
    model_dtype_name='fp32',
    seed=42,
)

# Original grad-norm proxy experiment keep ratios: top 1%, top 10%, top 50%, top 100%.
SPARSE_KEEP_RATIOS: Tuple[float, ...] = (0.1, 0.2, 0.5)

# Bounded sample size for global threshold estimation to avoid flattening full model vectors.
THRESHOLD_SAMPLE_SIZE = 2_000_000

# Keep original output namespace for grad-based sparse artifacts.
SPARSE_OUTPUT_ROOT = RUNTIME.output_root / 'if_sparse_topk_task_vector_gradnorm'
SUMMARY_PATH = RUNTIME.output_root / 'metadata' / 'if_sparse_topk_task_vector_gradnorm_summary.json'

if not RUNTIME.if_model_path.exists():
    raise FileNotFoundError(f'IF checkpoint path does not exist: {RUNTIME.if_model_path}')

SPARSE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f'Base model: {RUNTIME.base_model_id}')
print(f'IF model path: {RUNTIME.if_model_path}')
print(f'Grad output root: {SPARSE_OUTPUT_ROOT}')
print(f'Grad summary path: {SUMMARY_PATH}')

Base model: Qwen/Qwen3-1.7B
IF model path: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface
Grad output root: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_task_vector_gradnorm
Grad summary path: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/metadata/if_sparse_topk_task_vector_gradnorm_summary.json


In [14]:
def now_iso() -> str:
    """Return a UTC ISO-8601 timestamp string.

    Returns:
        Timestamp string in `YYYY-MM-DDTHH:MM:SSZ` format.
    """

    return datetime.utcnow().isoformat(timespec='seconds') + 'Z'


def to_json_compatible(obj: Any) -> Any:
    """Recursively convert runtime objects into JSON-serializable values.

    Args:
        obj: Arbitrary object that may include runtime-only types (`Path`, `torch.dtype`).

    Returns:
        JSON-safe nested structure.
    """

    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, torch.dtype):
        return str(obj)
    if isinstance(obj, dict):
        return {str(key): to_json_compatible(value) for key, value in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_json_compatible(value) for value in obj]
    return obj


def save_json(payload: Mapping[str, Any], output_path: Path) -> None:
    """Save JSON payload with conversion for runtime-only object types.

    Args:
        payload: Mapping payload to serialize.
        output_path: Destination JSON path.

    Returns:
        None. File is written to disk.
    """

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open('w', encoding='utf-8') as file:
        json.dump(to_json_compatible(dict(payload)), file, indent=2, ensure_ascii=False)


def resolve_torch_dtype(dtype_name: str) -> torch.dtype:
    """Resolve a dtype alias into an actual torch dtype.

    Args:
        dtype_name: One of `bf16`, `fp16`, or `fp32`.

    Returns:
        Torch dtype object.
    """

    lookup = {
        'bf16': torch.bfloat16,
        'fp16': torch.float16,
        'fp32': torch.float32,
    }
    if dtype_name not in lookup:
        raise ValueError(f'Unsupported dtype: {dtype_name}')
    return lookup[dtype_name]


def load_tokenizer_with_mistral_regex_fix(model_name_or_path: str) -> AutoTokenizer:
    """Load tokenizer with optional compatibility argument.

    Args:
        model_name_or_path: HF id or local checkpoint directory.

    Returns:
        Loaded tokenizer instance.
    """

    try:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
            fix_mistral_regex=True,
        )
    except TypeError:
        return AutoTokenizer.from_pretrained(
            model_name_or_path,
            trust_remote_code=True,
        )


def load_causal_lm(
    model_name_or_path: str | Path,
    torch_dtype: torch.dtype,
    device: str,
) -> tuple[AutoModelForCausalLM, AutoTokenizer]:
    """Load causal LM and tokenizer on the requested device.

    Args:
        model_name_or_path: HF id or local checkpoint path.
        torch_dtype: Loading dtype.
        device: Device string (typically `cpu` for this notebook).

    Returns:
        Tuple of `(model, tokenizer)`.
    """

    resolved = str(model_name_or_path)
    model = AutoModelForCausalLM.from_pretrained(
        resolved,
        torch_dtype=torch_dtype,
        device_map=None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model.to(device)
    model.eval()

    tokenizer = load_tokenizer_with_mistral_regex_fix(resolved)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
) -> None:
    """Validate that base and IF checkpoints expose identical parameter layouts.

    Args:
        base_model: Base checkpoint model.
        if_model: IF checkpoint model.

    Returns:
        None. Raises `ValueError` on incompatible parameter names/shapes.
    """

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    if set(base_named.keys()) != set(if_named.keys()):
        missing_in_if = sorted(set(base_named.keys()) - set(if_named.keys()))
        missing_in_base = sorted(set(if_named.keys()) - set(base_named.keys()))
        raise ValueError(
            'Parameter key mismatch between base and IF models. ' 
            f'missing_in_if={missing_in_if[:5]}, missing_in_base={missing_in_base[:5]}'
        )

    for param_name, base_param in base_named.items():
        if if_named[param_name].shape != base_param.shape:
            raise ValueError(
                f"Shape mismatch at '{param_name}': "
                f"base={tuple(base_param.shape)}, if={tuple(if_named[param_name].shape)}"
            )


def build_if_task_vector_gradnorm_scores(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
) -> Dict[str, torch.Tensor]:
    """Build coordinate-wise IF task-vector grad-norm proxy scores.

    Score definition:
        score_j = |Delta_if_j|
        Delta_if_j = theta_if_j - theta_base_j

    Rationale:
        For scalar coordinates, L2 norm magnitude and absolute value are identical.
        Using `|Delta_if|` gives a direct coordinate-level update-strength signal
        for global top-k filtering.

    Args:
        base_model: Base model that defines `theta_base`.
        if_model: IF model that defines `theta_if`.

    Returns:
        Mapping `parameter_name -> score_tensor` on CPU float32.
    """

    validate_parameter_compatibility(base_model=base_model, if_model=if_model)

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())
    score_tensors: Dict[str, torch.Tensor] = {}

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            # Only floating-point parameters participate in delta-space updates.
            if not torch.is_floating_point(base_param.data):
                continue

            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_named[param_name].data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            # Store absolute delta as coordinate-level grad-norm proxy.
            score_tensors[param_name] = delta_if.abs().cpu()

    return score_tensors


def count_total_elements(score_tensors: Mapping[str, torch.Tensor]) -> int:
    """Count total scalar coordinates across score tensors.

    Args:
        score_tensors: Parameter-score mapping.

    Returns:
        Total scalar coordinate count.
    """

    total_numel = 0
    for score_tensor in score_tensors.values():
        total_numel += int(score_tensor.numel())
    return int(total_numel)


def sample_global_score_values(
    score_tensors: Mapping[str, torch.Tensor],
    total_numel: int,
    sample_size: int,
    seed: int,
) -> np.ndarray:
    """Sample score values uniformly over global coordinate index space.

    Why this implementation exists:
        Fully concatenating all model coordinates into one giant vector is memory-heavy.
        This index-based streaming sample estimates global quantiles efficiently.

    Args:
        score_tensors: Parameter-score mapping.
        total_numel: Total number of scalar coordinates.
        sample_size: Number of sampled coordinates for threshold estimation.
        seed: RNG seed for reproducible sampling.

    Returns:
        1D NumPy array of sampled score values.
    """

    if total_numel <= 0 or sample_size <= 0:
        return np.zeros(0, dtype=np.float32)

    effective_sample_size = int(min(sample_size, total_numel))
    rng = np.random.default_rng(seed=seed)

    # Draw global coordinate indices with replacement, then stream through tensors once.
    sampled_global_indices = np.sort(
        rng.integers(low=0, high=total_numel, size=effective_sample_size, dtype=np.int64)
    )

    sampled_values = np.empty(effective_sample_size, dtype=np.float32)
    write_cursor = 0
    tensor_offset = 0

    for score_tensor in score_tensors.values():
        score_flat = score_tensor.detach().to(torch.float32).reshape(-1)
        tensor_numel = int(score_flat.numel())

        left = np.searchsorted(sampled_global_indices, tensor_offset, side='left')
        right = np.searchsorted(sampled_global_indices, tensor_offset + tensor_numel, side='left')

        if right > left:
            local_indices = sampled_global_indices[left:right] - tensor_offset
            local_index_tensor = torch.from_numpy(local_indices.astype(np.int64))
            local_values = score_flat.index_select(dim=0, index=local_index_tensor)

            next_cursor = write_cursor + (right - left)
            sampled_values[write_cursor:next_cursor] = local_values.cpu().numpy()
            write_cursor = next_cursor

        tensor_offset += tensor_numel

    if write_cursor != effective_sample_size:
        raise RuntimeError(
            f'Sampling bookkeeping mismatch: expected {effective_sample_size}, got {write_cursor}'
        )

    return sampled_values


def estimate_global_score_threshold(
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
    sample_size: int,
    seed: int,
) -> Tuple[float, int, int]:
    """Estimate a global score threshold that keeps the top ratio of coordinates.

    Args:
        score_tensors: Parameter-score mapping.
        keep_ratio: Target keep ratio in `(0, 1]`.
        sample_size: Number of sampled values used for quantile estimation.
        seed: RNG seed for sampling.

    Returns:
        Tuple `(threshold, total_numel, sampled_count)`.
    """

    if keep_ratio <= 0.0 or keep_ratio > 1.0:
        raise ValueError(f'keep_ratio must be in (0, 1], got {keep_ratio}')

    total_numel = count_total_elements(score_tensors=score_tensors)
    sampled_values = sample_global_score_values(
        score_tensors=score_tensors,
        total_numel=total_numel,
        sample_size=sample_size,
        seed=seed,
    )

    if sampled_values.size == 0:
        return 0.0, int(total_numel), int(sampled_values.size)

    # Top keep_ratio corresponds to quantile (1 - keep_ratio).
    threshold = float(np.quantile(sampled_values, q=(1.0 - float(keep_ratio))))
    return threshold, int(total_numel), int(sampled_values.size)


def apply_sparse_if_update_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
    score_threshold: float,
) -> Dict[str, Any]:
    """Apply sparse IF task-vector update to the base model in-place.

    Update rule:
        theta_sparse = theta_base + 1[score >= threshold] * (theta_if - theta_base)

    Args:
        base_model: Base model to overwrite with sparse-updated weights.
        if_model: IF model that provides dense IF delta.
        score_tensors: Parameter-score mapping used for coordinate masking.
        score_threshold: Global threshold for top-k coordinate retention.

    Returns:
        Summary dictionary with threshold and realized retention statistics.
    """

    validate_parameter_compatibility(base_model=base_model, if_model=if_model)

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    kept_elements = 0
    total_elements = 0

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            if not torch.is_floating_point(base_param.data):
                continue

            if param_name not in score_tensors:
                raise ValueError(f'Missing score tensor for parameter: {param_name}')

            score = score_tensors[param_name].detach().to(torch.float32)
            if tuple(score.shape) != tuple(base_param.shape):
                raise ValueError(
                    f"Score shape mismatch for '{param_name}': "
                    f"score={tuple(score.shape)} vs model={tuple(base_param.shape)}"
                )

            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_named[param_name].data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            # Keep only high-score coordinates and restore all others to base.
            sparse_mask = score >= float(score_threshold)
            sparse_update = delta_if * sparse_mask.to(torch.float32)
            merged_tensor = base_fp32 + sparse_update
            base_param.data.copy_(merged_tensor.to(base_param.dtype))

            kept_elements += int(sparse_mask.sum().item())
            total_elements += int(sparse_mask.numel())

    return {
        'score_threshold': float(score_threshold),
        'kept_elements': int(kept_elements),
        'total_elements': int(total_elements),
        'realized_keep_ratio': float(kept_elements / max(total_elements, 1)),
    }


def keep_ratio_to_tag(keep_ratio: float) -> str:
    """Convert keep ratio to a filesystem-safe tag string.

    Args:
        keep_ratio: Keep ratio in `(0, 1]`.

    Returns:
        Tag like `top_1pct` or `top_10pct`.
    """

    pct_value = keep_ratio * 100.0
    pct_str = f'{pct_value:.3f}'.rstrip('0').rstrip('.')
    return f"top_{pct_str.replace('.', 'p')}pct"


def save_sparse_if_checkpoint(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    output_dir: Path,
    metadata: Mapping[str, Any],
) -> None:
    """Save sparse IF checkpoint and metadata.

    Args:
        model: Sparse-updated model.
        tokenizer: Tokenizer saved together with the model.
        output_dir: Output checkpoint directory.
        metadata: JSON-serializable metadata payload.

    Returns:
        None. Artifacts are written to `output_dir`.
    """

    output_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(output_dir, safe_serialization=True)
    tokenizer.save_pretrained(output_dir)
    save_json(metadata, output_dir / 'merge_metadata.json')


def compute_exact_topk_threshold_info(
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
) -> Dict[str, Any]:
    """Compute exact global top-k threshold metadata for a keep ratio.

    Why this exists:
        Quantile-threshold masking can deviate from requested keep ratios because of
        sampling error and tied values near threshold. This helper computes an exact
        global top-k target count and deterministic threshold/tie budget information.

    Args:
        score_tensors: Mapping from parameter name to coordinate-wise score tensors.
        keep_ratio: Requested keep ratio in `[0, 1]`.

    Returns:
        Dictionary with exact top-k selection metadata:
        - `score_threshold`: Global threshold value used for `>` and `==` tie split.
        - `target_keep_elements`: Exact number of coordinates to keep.
        - `total_numel`: Total scalar coordinate count.
        - `strictly_greater_count`: Count of coordinates where `score > threshold`.
        - `equal_to_threshold_count`: Count of coordinates where `score == threshold`.
        - `equal_selection_budget`: Number of `== threshold` coordinates to keep.
        - `selection_mode`: Constant string `exact_global_topk`.

    Raises:
        ValueError: If `keep_ratio` is outside `[0, 1]`.
        RuntimeError: If threshold bookkeeping is inconsistent.
    """

    if keep_ratio < 0.0 or keep_ratio > 1.0:
        raise ValueError(f'keep_ratio must be in [0, 1], got {keep_ratio}')

    total_numel = count_total_elements(score_tensors=score_tensors)
    # We round to the nearest integer so requested ratio maps to the closest feasible
    # integer keep count on finite coordinate sets.
    target_keep_elements = int(round(float(keep_ratio) * float(total_numel)))
    target_keep_elements = int(max(0, min(target_keep_elements, total_numel)))

    if total_numel == 0:
        return {
            'score_threshold': float('inf'),
            'target_keep_elements': 0,
            'total_numel': 0,
            'strictly_greater_count': 0,
            'equal_to_threshold_count': 0,
            'equal_selection_budget': 0,
            'selection_mode': 'exact_global_topk',
        }

    if target_keep_elements == 0:
        return {
            'score_threshold': float('inf'),
            'target_keep_elements': 0,
            'total_numel': int(total_numel),
            'strictly_greater_count': 0,
            'equal_to_threshold_count': int(total_numel),
            'equal_selection_budget': 0,
            'selection_mode': 'exact_global_topk',
        }

    if target_keep_elements == total_numel:
        return {
            'score_threshold': float('-inf'),
            'target_keep_elements': int(target_keep_elements),
            'total_numel': int(total_numel),
            'strictly_greater_count': int(total_numel),
            'equal_to_threshold_count': 0,
            'equal_selection_budget': 0,
            'selection_mode': 'exact_global_topk',
        }

    # Exact threshold computation requires a full global view. This is memory-heavy
    # but guarantees exact top-k bookkeeping on tied coordinates.
    flattened_scores = torch.cat(
        [tensor.detach().to(torch.float32).reshape(-1) for tensor in score_tensors.values()],
        dim=0,
    )

    kth_smallest_rank = int(total_numel - target_keep_elements + 1)
    threshold = float(torch.kthvalue(flattened_scores, k=kth_smallest_rank).values.item())

    strictly_greater_count = int((flattened_scores > threshold).sum().item())
    equal_to_threshold_count = int((flattened_scores == threshold).sum().item())
    equal_selection_budget = int(target_keep_elements - strictly_greater_count)

    del flattened_scores
    gc.collect()

    if equal_selection_budget < 0 or equal_selection_budget > equal_to_threshold_count:
        raise RuntimeError(
            'Exact top-k threshold bookkeeping failed: '
            f'equal_selection_budget={equal_selection_budget}, '
            f'equal_to_threshold_count={equal_to_threshold_count}, '
            f'strictly_greater_count={strictly_greater_count}, '
            f'target_keep_elements={target_keep_elements}'
        )

    return {
        'score_threshold': float(threshold),
        'target_keep_elements': int(target_keep_elements),
        'total_numel': int(total_numel),
        'strictly_greater_count': int(strictly_greater_count),
        'equal_to_threshold_count': int(equal_to_threshold_count),
        'equal_selection_budget': int(equal_selection_budget),
        'selection_mode': 'exact_global_topk',
    }


def promote_first_true_entries_inplace(
    destination_mask_flat: torch.Tensor,
    candidate_true_flat: torch.Tensor,
    max_new_entries: int,
    chunk_size: int = 4_000_000,
) -> int:
    """Promote first `max_new_entries` True entries from candidate mask into destination.

    Why this exists:
        For large tied sets (`score == threshold`), materializing all tie indices via
        one full `nonzero` can be memory-heavy. This chunked routine keeps memory
        bounded while preserving deterministic first-seen ordering.

    Args:
        destination_mask_flat: 1D boolean tensor updated in-place.
        candidate_true_flat: 1D boolean tensor indicating selectable entries.
        max_new_entries: Maximum number of entries to promote.
        chunk_size: Number of coordinates processed per chunk.

    Returns:
        Number of entries actually promoted.
    """

    if max_new_entries <= 0:
        return 0

    promoted = 0
    total_numel = int(candidate_true_flat.numel())

    for start in range(0, total_numel, int(chunk_size)):
        if promoted >= max_new_entries:
            break

        end = min(start + int(chunk_size), total_numel)
        candidate_chunk = candidate_true_flat[start:end]
        candidate_count = int(candidate_chunk.sum().item())
        if candidate_count == 0:
            continue

        remaining_budget = int(max_new_entries - promoted)
        destination_chunk = destination_mask_flat[start:end]

        if candidate_count <= remaining_budget:
            # Entire chunk can be promoted without expensive index extraction.
            destination_chunk |= candidate_chunk
            promoted += candidate_count
            continue

        # Partial chunk selection: pick earliest `remaining_budget` true entries.
        local_true_indices = torch.nonzero(candidate_chunk, as_tuple=False).reshape(-1)
        take_count = int(min(remaining_budget, local_true_indices.numel()))
        if take_count > 0:
            destination_chunk[local_true_indices[:take_count]] = True
            promoted += take_count
        break

    return int(promoted)


def apply_sparse_if_update_exact_topk_inplace(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratio: float,
) -> Dict[str, Any]:
    """Apply sparse IF update using exact global top-k score selection.

    Update rule:
        theta_sparse = theta_base + mask_topk * (theta_if - theta_base)

    Selection rule (exact):
        1) Keep all coordinates with `score > threshold`.
        2) Keep first `equal_selection_budget` coordinates among `score == threshold`.
        3) This guarantees exact `target_keep_elements` globally.

    Args:
        base_model: Base model to overwrite with sparse-updated weights.
        if_model: IF model that provides dense IF delta.
        score_tensors: Parameter-score mapping used for global top-k selection.
        keep_ratio: Requested global keep ratio in `[0, 1]`.

    Returns:
        Summary dictionary with exact top-k metadata and realized keep statistics.
    """

    validate_parameter_compatibility(base_model=base_model, if_model=if_model)

    base_named = dict(base_model.named_parameters())
    if_named = dict(if_model.named_parameters())

    topk_info = compute_exact_topk_threshold_info(
        score_tensors=score_tensors,
        keep_ratio=keep_ratio,
    )
    threshold = float(topk_info['score_threshold'])
    target_keep_elements = int(topk_info['target_keep_elements'])
    total_numel = int(topk_info['total_numel'])
    remaining_equal_budget = int(topk_info['equal_selection_budget'])

    kept_elements = 0
    total_elements = 0

    with torch.no_grad():
        for param_name, base_param in base_named.items():
            if not torch.is_floating_point(base_param.data):
                continue

            if param_name not in score_tensors:
                raise ValueError(f'Missing score tensor for parameter: {param_name}')

            score = score_tensors[param_name].detach().to(torch.float32)
            if tuple(score.shape) != tuple(base_param.shape):
                raise ValueError(
                    f"Score shape mismatch for '{param_name}': "
                    f"score={tuple(score.shape)} vs model={tuple(base_param.shape)}"
                )

            base_fp32 = base_param.data.detach().to(torch.float32)
            if_fp32 = if_named[param_name].data.detach().to(torch.float32)
            delta_if = if_fp32 - base_fp32

            if target_keep_elements == 0:
                final_mask = torch.zeros_like(score, dtype=torch.bool)
            elif target_keep_elements == total_numel:
                final_mask = torch.ones_like(score, dtype=torch.bool)
            else:
                # Exact top-k split: strict-above threshold + deterministic ties.
                greater_mask = score > threshold
                final_mask = greater_mask.clone()

                if remaining_equal_budget > 0:
                    equal_mask_flat = (score == threshold).reshape(-1)
                    promoted_here = promote_first_true_entries_inplace(
                        destination_mask_flat=final_mask.reshape(-1),
                        candidate_true_flat=equal_mask_flat,
                        max_new_entries=int(remaining_equal_budget),
                    )
                    remaining_equal_budget -= int(promoted_here)

            sparse_update = delta_if * final_mask.to(torch.float32)
            merged_tensor = base_fp32 + sparse_update
            base_param.data.copy_(merged_tensor.to(base_param.dtype))

            kept_elements += int(final_mask.sum().item())
            total_elements += int(final_mask.numel())

    if remaining_equal_budget != 0:
        raise RuntimeError(
            'Exact top-k tie budget was not fully consumed: '
            f'remaining_equal_budget={remaining_equal_budget}'
        )

    if kept_elements != target_keep_elements:
        raise RuntimeError(
            'Exact top-k keep-count mismatch: '
            f'kept_elements={kept_elements}, target_keep_elements={target_keep_elements}'
        )

    return {
        **topk_info,
        'keep_ratio': float(keep_ratio),
        'kept_elements': int(kept_elements),
        'total_elements': int(total_elements),
        'realized_keep_ratio': float(kept_elements / max(total_elements, 1)),
    }


In [15]:
# Original grad-based sparse reconstruction cell.
# Updated to exact global top-k selection so realized keep ratios match targets.

merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)

# Load IF model once because it is reused for all sparse variants.
if_model_for_delta, _ = load_causal_lm(
    model_name_or_path=RUNTIME.if_model_path,
    torch_dtype=merge_dtype,
    device='cpu',
)

# Build score tensors from IF task-vector magnitudes against the base anchor.
base_anchor_for_scores, _ = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device='cpu',
)
score_tensors = build_if_task_vector_gradnorm_scores(
    base_model=base_anchor_for_scores,
    if_model=if_model_for_delta,
)
del base_anchor_for_scores
gc.collect()

sparse_run_rows = []

for keep_ratio in SPARSE_KEEP_RATIOS:
    # Exact top-k application: target keep count is guaranteed by construction.
    sparse_model, sparse_tokenizer = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=merge_dtype,
        device='cpu',
    )

    sparse_summary = apply_sparse_if_update_exact_topk_inplace(
        base_model=sparse_model,
        if_model=if_model_for_delta,
        score_tensors=score_tensors,
        keep_ratio=keep_ratio,
    )

    output_tag = keep_ratio_to_tag(keep_ratio=keep_ratio)
    output_dir = SPARSE_OUTPUT_ROOT / f'if_delta_sparse_{output_tag}'

    metadata = {
        'created_at': now_iso(),
        'method': 'if_delta_sparse_by_task_vector_gradnorm_topk',
        'selection_mode': 'exact_global_topk',
        'formula': 'theta_sparse = theta_base + 1[|Delta_if| in global top-k] * Delta_if',
        'score_definition': '|Delta_if| where Delta_if = theta_if - theta_base',
        'base_model_id': str(RUNTIME.base_model_id),
        'if_model_path': str(RUNTIME.if_model_path),
        'keep_ratio': float(keep_ratio),
        'target_keep_elements': int(sparse_summary['target_keep_elements']),
        'topk_threshold': float(sparse_summary['score_threshold']),
        'sparse_summary': sparse_summary,
    }

    save_sparse_if_checkpoint(
        model=sparse_model,
        tokenizer=sparse_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    sparse_run_rows.append(
        {
            'keep_ratio': float(keep_ratio),
            'target_keep_elements': int(sparse_summary['target_keep_elements']),
            'score_threshold': float(sparse_summary['score_threshold']),
            'realized_keep_ratio': float(sparse_summary['realized_keep_ratio']),
            'kept_elements': int(sparse_summary['kept_elements']),
            'total_elements': int(sparse_summary['total_elements']),
            'output_dir': str(output_dir),
        }
    )

    print(
        f"Saved grad-based sparse IF checkpoint | keep_ratio={keep_ratio:.4f} "
        f"| target_keep={sparse_summary['target_keep_elements']} "
        f"| kept={sparse_summary['kept_elements']} "
        f"| realized={sparse_summary['realized_keep_ratio']:.6f} "
        f"| threshold={sparse_summary['score_threshold']:.6e} "
        f"| path={output_dir}"
    )

    del sparse_model
    del sparse_tokenizer
    gc.collect()

summary_payload = {
    'created_at': now_iso(),
    'runtime': asdict(RUNTIME),
    'method': 'if_delta_sparse_by_task_vector_gradnorm_topk',
    'selection_mode': 'exact_global_topk',
    'keep_ratios': [float(ratio) for ratio in SPARSE_KEEP_RATIOS],
    'runs': sparse_run_rows,
}
save_json(summary_payload, SUMMARY_PATH)

display(pd.DataFrame(sparse_run_rows))
print(f'Saved grad-based sparse IF run summary: {SUMMARY_PATH}')

del if_model_for_delta
del score_tensors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.18it/s]


Saved grad-based sparse IF checkpoint | keep_ratio=0.1000 | target_keep=172057498 | kept=172057498 | realized=0.100000 | threshold=2.214126e-05 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_task_vector_gradnorm/if_delta_sparse_top_10pct


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.43it/s]


Saved grad-based sparse IF checkpoint | keep_ratio=0.2000 | target_keep=344114995 | kept=344114995 | realized=0.200000 | threshold=1.689140e-05 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_task_vector_gradnorm/if_delta_sparse_top_20pct


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.30it/s]


Saved grad-based sparse IF checkpoint | keep_ratio=0.5000 | target_keep=860287488 | kept=860287488 | realized=0.500000 | threshold=7.964671e-06 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_task_vector_gradnorm/if_delta_sparse_top_50pct


,keep_ratio,target_keep_elements,score_threshold,realized_keep_ratio,kept_elements,total_elements,output_dir
0,0.1,172057498,0.000022,0.1,172057498,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...
1,0.2,344114995,0.000017,0.2,344114995,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...
2,0.5,860287488,0.000008,0.5,860287488,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...


Saved grad-based sparse IF run summary: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/metadata/if_sparse_topk_task_vector_gradnorm_summary.json


In [ ]:
# Appended Fisher-based sparse reconstruction cell (requested).
# This cell keeps the original grad-based cells unchanged and runs a separate
# Fisher-topk sparse update experiment with keep ratios 0.1%, 1%, 10%, 100%.
# Updated to exact global top-k selection to guarantee requested keep ratios.

FISHER_IF_PATH = Path('/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/fisher_diagonal/fisher_diag_if.pt')
FISHER_KEEP_RATIOS: Tuple[float, ...] = (0.001, 0.005, 0.01, 0.05, 0.10, 0.20, 0.50, 1.00)
FISHER_OUTPUT_ROOT = RUNTIME.output_root / 'if_sparse_topk_fisher_0p1_1_10_100'
FISHER_SUMMARY_PATH = RUNTIME.output_root / 'metadata' / 'if_sparse_topk_fisher_summary_0p1_1_10_100.json'

if not FISHER_IF_PATH.exists():
    raise FileNotFoundError(f'IF fisher path does not exist: {FISHER_IF_PATH}')

FISHER_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FISHER_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

def load_if_fisher_as_score_tensors(fisher_path: Path) -> Dict[str, torch.Tensor]:
    """Load IF Fisher diagonal artifact and convert to score tensors.

    Args:
        fisher_path: Path to serialized IF Fisher dictionary (`.pt`).

    Returns:
        Mapping from parameter name to absolute Fisher values on CPU float32.

    Raises:
        FileNotFoundError: If Fisher artifact is missing.
        TypeError: If loaded object is not a mapping.
    """

    if not fisher_path.exists():
        raise FileNotFoundError(f'IF Fisher file was not found: {fisher_path}')

    fisher_obj = torch.load(fisher_path, map_location='cpu')
    if not isinstance(fisher_obj, Mapping):
        raise TypeError(f'IF Fisher file must contain a mapping, got {type(fisher_obj)}')

    score_tensors: Dict[str, torch.Tensor] = {}
    for parameter_name, fisher_tensor in fisher_obj.items():
        # Use absolute Fisher values defensively in case tiny numerical negatives exist.
        score_tensors[str(parameter_name)] = fisher_tensor.detach().to(torch.float32).abs().cpu()

    return score_tensors


def validate_fisher_score_compatibility(
    base_model: AutoModelForCausalLM,
    score_tensors: Mapping[str, torch.Tensor],
) -> None:
    """Validate Fisher score tensor key/shape compatibility with base model.

    Args:
        base_model: Base model providing reference parameter names/shapes.
        score_tensors: Fisher-based score tensor mapping.

    Returns:
        None. Raises `ValueError` when keys/shapes are incompatible.
    """

    for parameter_name, base_param in base_model.named_parameters():
        if not torch.is_floating_point(base_param.data):
            continue
        if parameter_name not in score_tensors:
            raise ValueError(f'Missing Fisher score tensor for parameter: {parameter_name}')
        if tuple(score_tensors[parameter_name].shape) != tuple(base_param.shape):
            raise ValueError(
                f"Fisher score shape mismatch for '{parameter_name}': "
                f"score={tuple(score_tensors[parameter_name].shape)} vs model={tuple(base_param.shape)}"
            )

merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)

# Load IF model once for dense IF delta source in sparse update formula.
if_model_for_delta, _ = load_causal_lm(
    model_name_or_path=RUNTIME.if_model_path,
    torch_dtype=merge_dtype,
    device='cpu',
)

# Load one base model instance for compatibility checks.
base_model_for_validation, _ = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device='cpu',
)
validate_parameter_compatibility(base_model=base_model_for_validation, if_model=if_model_for_delta)

fisher_score_tensors = load_if_fisher_as_score_tensors(fisher_path=FISHER_IF_PATH)
validate_fisher_score_compatibility(
    base_model=base_model_for_validation,
    score_tensors=fisher_score_tensors,
)
del base_model_for_validation
gc.collect()

fisher_sparse_rows = []

for keep_ratio in FISHER_KEEP_RATIOS:
    sparse_model, sparse_tokenizer = load_causal_lm(
        model_name_or_path=RUNTIME.base_model_id,
        torch_dtype=merge_dtype,
        device='cpu',
    )

    sparse_summary = apply_sparse_if_update_exact_topk_inplace(
        base_model=sparse_model,
        if_model=if_model_for_delta,
        score_tensors=fisher_score_tensors,
        keep_ratio=keep_ratio,
    )

    output_tag = keep_ratio_to_tag(keep_ratio=keep_ratio)
    output_dir = FISHER_OUTPUT_ROOT / f'if_delta_sparse_fisher_{output_tag}'

    metadata = {
        'created_at': now_iso(),
        'method': 'if_delta_sparse_by_if_fisher_topk',
        'selection_mode': 'exact_global_topk',
        'formula': 'theta_sparse = theta_base + 1[F_if in global top-k] * (theta_if - theta_base)',
        'score_definition': 'F_if (absolute fisher diagonal)',
        'base_model_id': str(RUNTIME.base_model_id),
        'if_model_path': str(RUNTIME.if_model_path),
        'if_fisher_path': str(FISHER_IF_PATH),
        'keep_ratio': float(keep_ratio),
        'target_keep_elements': int(sparse_summary['target_keep_elements']),
        'topk_threshold': float(sparse_summary['score_threshold']),
        'sparse_summary': {
            'fisher_threshold': float(sparse_summary['score_threshold']),
            'target_keep_elements': int(sparse_summary['target_keep_elements']),
            'kept_elements': int(sparse_summary['kept_elements']),
            'total_elements': int(sparse_summary['total_elements']),
            'realized_keep_ratio': float(sparse_summary['realized_keep_ratio']),
            'selection_mode': str(sparse_summary['selection_mode']),
        },
    }

    save_sparse_if_checkpoint(
        model=sparse_model,
        tokenizer=sparse_tokenizer,
        output_dir=output_dir,
        metadata=metadata,
    )

    fisher_sparse_rows.append(
        {
            'keep_ratio': float(keep_ratio),
            'target_keep_elements': int(sparse_summary['target_keep_elements']),
            'fisher_threshold': float(sparse_summary['score_threshold']),
            'realized_keep_ratio': float(sparse_summary['realized_keep_ratio']),
            'kept_elements': int(sparse_summary['kept_elements']),
            'total_elements': int(sparse_summary['total_elements']),
            'output_dir': str(output_dir),
        }
    )

    print(
        f"Saved fisher-based sparse IF checkpoint | keep_ratio={keep_ratio:.4f} "
        f"| target_keep={sparse_summary['target_keep_elements']} "
        f"| kept={sparse_summary['kept_elements']} "
        f"| realized={sparse_summary['realized_keep_ratio']:.6f} "
        f"| threshold={sparse_summary['score_threshold']:.6e} "
        f"| path={output_dir}"
    )

    del sparse_model
    del sparse_tokenizer
    gc.collect()

fisher_summary_payload = {
    'created_at': now_iso(),
    'runtime': asdict(RUNTIME),
    'method': 'if_delta_sparse_by_if_fisher_topk',
    'selection_mode': 'exact_global_topk',
    'if_fisher_path': str(FISHER_IF_PATH),
    'keep_ratios': [float(ratio) for ratio in FISHER_KEEP_RATIOS],
    'runs': fisher_sparse_rows,
}
save_json(fisher_summary_payload, FISHER_SUMMARY_PATH)

display(pd.DataFrame(fisher_sparse_rows))
print(f'Saved fisher-based sparse IF run summary: {FISHER_SUMMARY_PATH}')

del if_model_for_delta
del fisher_score_tensors
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.55it/s]


Saved fisher-based sparse IF checkpoint | keep_ratio=0.0010 | target_keep=1720575 | kept=1720575 | realized=0.001000 | threshold=2.104220e+00 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_fisher_0p1_1_10_100/if_delta_sparse_fisher_top_0p1pct


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.57it/s]


Saved fisher-based sparse IF checkpoint | keep_ratio=0.0100 | target_keep=17205750 | kept=17205750 | realized=0.010000 | threshold=1.783354e-01 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_fisher_0p1_1_10_100/if_delta_sparse_fisher_top_1pct


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.87it/s]


Saved fisher-based sparse IF checkpoint | keep_ratio=0.1000 | target_keep=172057498 | kept=172057498 | realized=0.100000 | threshold=2.334415e-02 | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_fisher_0p1_1_10_100/if_delta_sparse_fisher_top_10pct


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.81it/s]


Saved fisher-based sparse IF checkpoint | keep_ratio=1.0000 | target_keep=1720574976 | kept=1720574976 | realized=1.000000 | threshold=-inf | path=/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/if_sparse_topk_fisher_0p1_1_10_100/if_delta_sparse_fisher_top_100pct


,keep_ratio,target_keep_elements,fisher_threshold,realized_keep_ratio,kept_elements,total_elements,output_dir
0,0.001,1720575,2.104220,0.001,1720575,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...
1,0.010,17205750,0.178335,0.010,17205750,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...
2,0.100,172057498,0.023344,0.100,172057498,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...
3,1.000,1720574976,-inf,1.000,1720574976,1720574976,/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Q...


Saved fisher-based sparse IF run summary: /mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-fisher-merge/metadata/if_sparse_topk_fisher_summary_0p1_1_10_100.json


## Task-Vector Score Diagnostics (Visualization)

This appended section visualizes IF task-vector coordinate scores `|Δ_if|` with:

1. Score histogram focused near zero
2. CDF-style curve of **top x% coordinate share** of global update norm energy
3. Exact zero-coordinate ratio (`|Δ_if| == 0`)
4. Estimated threshold table for each configured keep ratio

All computations reuse existing helper functions and keep the original sparse-update cells unchanged.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')

def compute_exact_zero_statistics(score_tensors: Mapping[str, torch.Tensor]) -> Dict[str, float]:
    """Compute exact zero-coordinate statistics from score tensors.

    Why this exists:
        The user requested exact `|Δ_if| == 0` coverage. This function iterates
        tensor-by-tensor without flattening everything into one giant vector,
        so memory usage stays bounded for large models.

    Args:
        score_tensors: Mapping of `parameter_name -> |Δ_if|` tensors.

    Returns:
        Dictionary with `zero_count`, `total_count`, and `zero_ratio`.
    """

    zero_count = 0
    total_count = 0

    for score_tensor in score_tensors.values():
        flat_scores = score_tensor.detach().to(torch.float32).reshape(-1)
        zero_count += int((flat_scores == 0.0).sum().item())
        total_count += int(flat_scores.numel())

    return {
        'zero_count': float(zero_count),
        'total_count': float(total_count),
        'zero_ratio': float(zero_count / max(total_count, 1)),
    }


def compute_topk_l2_energy_curve(sampled_scores: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Compute top-k coordinate share curve using sampled L2 energy.

    Definition:
        score_j = |Δ_if_j|
        energy_j = score_j^2
        top-k share = sum_{j in top-k}(energy_j) / sum_j(energy_j)

    Why this metric:
        The request asked for how much of the overall norm is captured by top x%.
        L2 norm contribution is naturally represented by squared magnitudes.

    Args:
        sampled_scores: 1D sampled score array (`|Δ_if|`).

    Returns:
        Tuple `(x_percent, cumulative_energy_percent)` where:
        - `x_percent`: top-coordinate fraction in percent
        - `cumulative_energy_percent`: captured L2 energy in percent
    """

    if sampled_scores.size == 0:
        return np.array([0.0, 100.0]), np.array([0.0, 100.0])

    energies = np.square(sampled_scores.astype(np.float64))
    total_energy = float(energies.sum())
    if total_energy <= 0.0:
        return np.array([0.0, 100.0]), np.array([0.0, 100.0])

    # Sort descending so cumulative sum traces top-k capture behavior.
    sorted_energy_desc = np.sort(energies)[::-1]
    cumulative_energy = np.cumsum(sorted_energy_desc)

    x_percent = (np.arange(1, sorted_energy_desc.size + 1, dtype=np.float64) / sorted_energy_desc.size) * 100.0
    cumulative_energy_percent = (cumulative_energy / total_energy) * 100.0

    return x_percent, cumulative_energy_percent


def summarize_topk_energy_anchors(
    x_percent: np.ndarray,
    cumulative_energy_percent: np.ndarray,
    anchor_top_percents: Tuple[float, ...],
) -> pd.DataFrame:
    """Summarize top-k energy capture at anchor percentages.

    Args:
        x_percent: X-axis percentages from `compute_topk_l2_energy_curve`.
        cumulative_energy_percent: Y-axis percentages from the same curve.
        anchor_top_percents: Anchor top-k percentages to report (e.g., 0.1, 1, 10).

    Returns:
        DataFrame with approximate captured L2-energy percentages at each anchor.
    """

    rows = []
    for anchor in anchor_top_percents:
        # `searchsorted` gives first index where x >= anchor for monotonic arrays.
        idx = int(np.searchsorted(x_percent, anchor, side='left'))
        idx = min(max(idx, 0), max(len(x_percent) - 1, 0))
        captured = float(cumulative_energy_percent[idx]) if len(cumulative_energy_percent) > 0 else float('nan')
        rows.append({
            'top_percent_coordinates': float(anchor),
            'captured_l2_energy_percent': captured,
        })
    return pd.DataFrame(rows)


def build_threshold_table_for_keep_ratios(
    score_tensors: Mapping[str, torch.Tensor],
    keep_ratios: Tuple[float, ...],
    sample_size: int,
    seed: int,
) -> pd.DataFrame:
    """Build exact top-k threshold table for each keep ratio.

    Why `sample_size` and `seed` remain in signature:
        This function keeps API compatibility with previous notebook calls. In exact
        top-k mode they are informational only and not used by the computation.

    Args:
        score_tensors: Mapping of `parameter_name -> |Δ_if|` tensors.
        keep_ratios: Keep-ratio tuple to evaluate (e.g., `(0.01, 0.10, ...)`).
        sample_size: Legacy argument retained for compatibility.
        seed: Legacy argument retained for compatibility.

    Returns:
        DataFrame with exact top-k thresholds and target keep counts.
    """

    rows = []
    for keep_ratio in keep_ratios:
        topk_info = compute_exact_topk_threshold_info(
            score_tensors=score_tensors,
            keep_ratio=keep_ratio,
        )
        rows.append({
            'keep_ratio': float(keep_ratio),
            'keep_ratio_percent': float(keep_ratio * 100.0),
            'expected_threshold': float(topk_info['score_threshold']),
            'target_keep_elements': int(topk_info['target_keep_elements']),
            'total_coordinate_count': int(topk_info['total_numel']),
            'selection_mode': str(topk_info['selection_mode']),
            'legacy_threshold_sample_size': int(sample_size),
            'legacy_seed': int(seed),
        })
    return pd.DataFrame(rows).sort_values('keep_ratio').reset_index(drop=True)


# ----------------------------------------------------------------------------------
# Build task-vector scores (`|Δ_if|`) and sampled distribution for visualization.
# ----------------------------------------------------------------------------------
VIS_SAMPLE_SIZE = THRESHOLD_SAMPLE_SIZE
VIS_ENERGY_ANCHORS: Tuple[float, ...] = (0.1, 0.5, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0)

merge_dtype = resolve_torch_dtype(RUNTIME.model_dtype_name)

# Load IF and base models specifically for diagnostics so this cell is standalone-runnable.
if_model_vis, _ = load_causal_lm(
    model_name_or_path=RUNTIME.if_model_path,
    torch_dtype=merge_dtype,
    device='cpu',
)
base_model_vis, _ = load_causal_lm(
    model_name_or_path=RUNTIME.base_model_id,
    torch_dtype=merge_dtype,
    device='cpu',
)

score_tensors_vis = build_if_task_vector_gradnorm_scores(
    base_model=base_model_vis,
    if_model=if_model_vis,
)

total_numel_vis = count_total_elements(score_tensors=score_tensors_vis)
sampled_scores_vis = sample_global_score_values(
    score_tensors=score_tensors_vis,
    total_numel=total_numel_vis,
    sample_size=VIS_SAMPLE_SIZE,
    seed=RUNTIME.seed,
)

# ----------------------------------------------------------------------------------
# 1) Score histogram near zero + sampled distribution diagnostics
# ----------------------------------------------------------------------------------
if sampled_scores_vis.size == 0:
    raise RuntimeError('Sampled score array is empty. Check score tensor construction.')

score_q90 = float(np.quantile(sampled_scores_vis, 0.90))
score_q99 = float(np.quantile(sampled_scores_vis, 0.99))
score_q999 = float(np.quantile(sampled_scores_vis, 0.999))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: broad view up to q99 so long-tail outliers do not collapse near-zero bins.
axes[0].hist(
    np.clip(sampled_scores_vis, a_min=0.0, a_max=score_q99),
    bins=180,
    color='steelblue',
    alpha=0.8,
)
axes[0].set_title('Score Histogram (clipped at q99)')
axes[0].set_xlabel('|Δ_if| score')
axes[0].set_ylabel('Sample count')
axes[0].axvline(score_q90, color='orange', linestyle='--', linewidth=1.5, label=f'q90={score_q90:.3e}')
axes[0].axvline(score_q99, color='red', linestyle='--', linewidth=1.5, label=f'q99={score_q99:.3e}')
axes[0].legend()

# Right panel: near-zero zoom to explicitly show concentration around zero.
near_zero_cap = max(score_q90, 1e-20)
axes[1].hist(
    np.clip(sampled_scores_vis, a_min=0.0, a_max=near_zero_cap),
    bins=180,
    color='darkorange',
    alpha=0.8,
)
axes[1].set_title('Near-Zero Zoom (0 ~ q90)')
axes[1].set_xlabel('|Δ_if| score')
axes[1].set_ylabel('Sample count')
axes[1].axvline(score_q999, color='purple', linestyle='--', linewidth=1.5, label=f'q99.9={score_q999:.3e}')
axes[1].legend()

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------------------
# 2) CDF-style top-x% norm share (L2 energy share curve)
# ----------------------------------------------------------------------------------
x_percent_vis, cdf_energy_percent_vis = compute_topk_l2_energy_curve(sampled_scores=sampled_scores_vis)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_percent_vis, cdf_energy_percent_vis, color='seagreen', linewidth=2.5)
ax.set_xscale('log')
ax.set_xlim(0.01, 100.0)
ax.set_ylim(0.0, 100.0)
ax.set_title('Top-x% Coordinates vs Captured L2 Energy Share')
ax.set_xlabel('Top coordinates kept (%) [log scale]')
ax.set_ylabel('Captured L2 energy (%)')
ax.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

energy_anchor_df = summarize_topk_energy_anchors(
    x_percent=x_percent_vis,
    cumulative_energy_percent=cdf_energy_percent_vis,
    anchor_top_percents=VIS_ENERGY_ANCHORS,
)
display(energy_anchor_df)

# ----------------------------------------------------------------------------------
# 3) Exact zero ratio (`|Δ_if| == 0`)
# ----------------------------------------------------------------------------------
zero_stats = compute_exact_zero_statistics(score_tensors=score_tensors_vis)
zero_stats_df = pd.DataFrame([
    {
        'zero_count': int(zero_stats['zero_count']),
        'total_count': int(zero_stats['total_count']),
        'zero_ratio': float(zero_stats['zero_ratio']),
        'zero_ratio_percent': float(zero_stats['zero_ratio'] * 100.0),
    }
])
display(zero_stats_df)

# ----------------------------------------------------------------------------------
# 4) Estimated threshold table for each keep ratio
# ----------------------------------------------------------------------------------
threshold_table_df = build_threshold_table_for_keep_ratios(
    score_tensors=score_tensors_vis,
    keep_ratios=SPARSE_KEEP_RATIOS,
    sample_size=THRESHOLD_SAMPLE_SIZE,
    seed=RUNTIME.seed,
)
display(threshold_table_df)

# Cleanup to keep notebook memory footprint stable for subsequent cells.
del base_model_vis
del if_model_vis
del score_tensors_vis
del sampled_scores_vis
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()